# NB30 — KANSER Panel Advanced Stacking: COMBINED + OOF Stacking + NN/DNN Base Models

KANSER panelinin %80/20 F1 skorunu NB16 referansının (0.716, CI=[0.67-0.77]) üzerine çıkarmak için, NB29'da başarısız olan stacking katmanını ekleyerek ileriye gidiyoruz.

**Motivasyon:**
- NB29 E1 (COMBINED pooling): Boot F1=0.7143 (yalnızca 0.0017 puan NB16'nın üzerine, hemen hemen eşit)
- NB29'da stacking katmanı yoktu → Şimdi NN/DNN + tree base modeller ile heterogeneous stacking deniyoruz
- NB16 bulgusu: **Meta-learner = Logistic Regression (L2, class_weight=balanced)** — GBM meta overfit eder
- NB15/NB16 buluşu: KANSER'de finetune-DNN kazanır (NN/DNN'in küçük panelerde katkısını ölç)

**5 Deney:**
| # | Deney | Açıklama |
|---|-------|----------|
| E0 | Baseline | KANSER-only LightGBM (OOF) — referans |
| E1 | COMBINED Baseline | MASTER+PAH+CFTR LightGBM → KANSER test (cross-panel, leakage yok) |
| E2 | COMBINED + OOF Stacking (Tree) | 4 base tree (LGBM, XGB, RF, CatBoost) + LR meta |
| E3 | COMBINED + NN/DNN Only Stacking | 2 base NN (SmallMLP, DeepMLP) + LR meta |
| E4 | Zengin Base Set (BalBag+5 model) | BalBag(LGBM) + RF + SmallMLP + LGBM + CatBoost + LR meta |
| E5 | KANSER-only NN/DNN (OOF) | NN/DNN'in küçük veri performansı (focal loss + early stopping) |

**Kritik Kurallar:**
1. **Leakage yok**: COMBINED = MASTER+PAH+CFTR (KANSER HARİÇ!). COMBINED'da eğitilip KANSER'de test.
2. **OOF ile self-prediction leakage önle**: KANSER-üzeri deneyler (E0, E5) 5-fold OOF kullan.
3. **Threshold seçimi**: `select_threshold_8020_robust(y, prob, n=50)` — N=50 bootstrap ortalaması.
4. **Birincil metrik**: Bootstrap %80/20 pathogenic-F1 (N=50). İkincil: LOO-MCC (prior-shift).
5. **Prior-shift**: Saerens 2002, pi_test=0.20. Kalibrasyon ÖNCE, prior-shift SONRA.
6. **Meta-learner = Logistic Regression** (L2, class_weight=balanced) — GBM meta overfit eder.
7. **is_missing_* flag (M3)**: >%50 NaN sütunlar için binary flag + medyan imputation.
8. **SEED=42** her yerde.
9. **NN/DNN torch inline**: Notebook-inline tanımla, src/models.py'ı import ETME.
10. **fpdf2 PDF rapor**: ASCII-safe Türkçe karakterler (ş→s, ğ→g vs.).

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# torch lazy import
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# imblearn
try:
    from imblearn.ensemble import BalancedBaggingClassifier
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("[UYARI] imblearn bulunamadı")

# catboost
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("[UYARI] catboost bulunamadı")

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, LeaveOneOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

np.random.seed(SEED)
torch.manual_seed(SEED)

# Sabitler
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123

TARGET_PANEL = "KANSER"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v15_kanser_advanced_stacking")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f"NB30 -- KANSER Advanced Stacking")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")
print(f"HAS_IMBLEARN={HAS_IMBLEARN}, HAS_CATBOOST={HAS_CATBOOST}")

NB30 -- KANSER Advanced Stacking
SEED=42, PI_TEST=0.2, N_BOOT=50
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v15_kanser_advanced_stacking
HAS_IMBLEARN=True, HAS_CATBOOST=True


In [2]:
# Cell 2: Veri Yükleme + Sütun Temizliği
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# --- Veri yükleme ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- COMBINED: MASTER+PAH+CFTR (KANSER HARİÇ!) ---
df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+PAH+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# --- Cross-panel birebir-aynı satır drop ---
feat_cols = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tüm feature+label birebir aynı olan satırların panel ID'lerini döndür."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"KANSER: {len(dup_ids)} birebir-aynı satır drop -> {df_kanser.shape}")
else:
    print("KANSER: birebir-aynı satır yok")

# --- Sütun temizliği: constant + duplicate ---
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

# Her dataset'ten drop
keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}")

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+PAH+CFTR): (3414, 353) (pos=2549, neg=865)
KANSER: 3 birebir-aynı satır drop -> (385, 353)
Constant: 0, Duplicate pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293

Final shapes: MASTER=(2931, 295), COMBINED=(3414, 295), KANSER=(385, 295)


In [3]:
# Cell 3: M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    """Train üzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit (train üzerinde)
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median (train üzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sütunlar için)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

# Preprocessor fit (MASTER + COMBINED)
prep_master = fit_preprocessor(df_master, keep_cols, TARGET)
prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

# Transform
X_master_df = transform_X(df_master, keep_cols, prep_master)
y_master = df_master[TARGET].values

X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_kanser_master_df = transform_X(df_kanser, keep_cols, prep_master)
X_kanser_combined_df = transform_X(df_kanser, keep_cols, prep_combined)
y_kanser = df_kanser[TARGET].values

print(f"X_master: {X_master_df.shape}, X_combined: {X_combined_df.shape}")
print(f"X_kanser: {X_kanser_master_df.shape} (master prep), {X_kanser_combined_df.shape} (combined prep)")
print(f"KANSER label dist: pos={y_kanser.sum()}, neg={(y_kanser==0).sum()}")

X_master: (2931, 434), X_combined: (3414, 434)
X_kanser: (385, 434) (master prep), (385, 434) (combined prep)
KANSER label dist: pos=265, neg=120


In [4]:
# Cell 4: Değerlendirme Altyapısı

# --- Prior shift (Saerens 2002) ---
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- Bootstrap %80/20 ---
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best:
            best, best_thr = f, thr
    return float(best_thr)

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """N resample ortalamasıyla robust threshold seç."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

# --- LOO-CV metrikleri ---
def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1 = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    auprc = average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "auprc": auprc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "y_pred": y_pred, "y_true": np.asarray(y_true), "prob": prob}

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Değerlendirme altyapısı hazır.")

Değerlendirme altyapısı hazır.


In [5]:
# Cell 5: Model Yardımcıları (Tree + Factory)

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbosity": 0,
    "n_jobs": -1
}

def make_lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def make_xgb_classifier(**kw):
    params = {**XGB_PARAMS, **kw}
    return XGBClassifier(**params)

def make_rf_classifier(**kw):
    params = {"n_estimators": 200, "max_depth": 15, "random_state": SEED, "n_jobs": -1}
    params.update(kw)
    return RandomForestClassifier(**params)

def make_catboost_classifier(**kw):
    params = {"iterations": 300, "verbose": 0, "random_state": SEED}
    params.update(kw)
    if HAS_CATBOOST:
        return CatBoostClassifier(**params)
    return None

def _le_encode_for_loo(X_df):
    """Kategorik sütunları basit label-encode et (LOO uyumlu)."""
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

def make_stack_meta_features(preds, prefix="base"):
    """Stacking meta-feature kolonlarını sklearn uyumlu string isimlerle üret."""
    preds = np.asarray(preds)
    meta = pd.DataFrame(preds, columns=[f"{prefix}_{i}" for i in range(preds.shape[1])])
    meta["mean"] = preds.mean(axis=1)
    meta["std"] = preds.std(axis=1)
    return meta

def oof_predict(make_model_fn, X_df, y, n_splits=5, seed=SEED):
    """Gerçek OOF tahmin: her örnek, onu görmemiş fold modelinden tahmin alır."""
    oof = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tri, vai in skf.split(X_df, y):
        m = make_model_fn()
        m.fit(X_df.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(X_df.iloc[vai])[:, 1]
    full = make_model_fn()
    full.fit(X_df, y)
    return oof, full

print("Model yardımcıları hazır.")

Model yardımcıları hazır.


In [6]:
# Cell 6: NN/DNN Modelleri (inline torch)

class SmallMLP(nn.Module):
    """NB16 reçetesi: BatchNorm YOK, yüksek dropout + weight_decay."""
    def __init__(self, input_dim, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class DeepMLP(nn.Module):
    """3 katman + residual, BatchNorm YOK."""
    def __init__(self, input_dim, hidden_dim=128, n_layers=3, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)
            ))
        self.output = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout * 0.5)
    def forward(self, x):
        x = self.dropout(self.relu(self.input_proj(x)))
        for i, block in enumerate(self.blocks):
            residual = x
            x = block(x)
            if i % 2 == 1:
                x = x + residual
        return self.output(x).squeeze(-1)

class FocalLoss(nn.Module):
    """Focal loss: benign'i öğrenmeye zorla (hard examples)."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()

def train_nn(model_class, X_train, y_train, X_val=None, y_val=None,
             hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4,
             epochs=200, patience=20, batch_size=64, **model_kw):
    """NN eğitimi: FocalLoss + early stopping."""
    device = torch.device("cpu")
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    
    model = model_class(X_tr_sc.shape[1], hidden_dim=hidden_dim, dropout=dropout, **model_kw)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    X_t = torch.FloatTensor(X_tr_sc).to(device)
    y_t = torch.FloatTensor(y_train).to(device)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    best_loss = float('inf')
    wait = 0
    best_state = None
    
    if X_val is not None:
        X_val_sc = scaler.transform(X_val)
        X_v = torch.FloatTensor(X_val_sc).to(device)
        y_v = torch.FloatTensor(y_val).to(device)
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            if X_val is not None:
                val_loss = criterion(model(X_v), y_v).item()
            else:
                val_loss = criterion(model(X_t), y_t).item()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model, scaler

def predict_nn(model, X, scaler):
    """NN tahmin: olasılık döndür."""
    device = torch.device("cpu")
    X_sc = scaler.transform(X) if scaler else X
    X_t = torch.FloatTensor(X_sc).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_t)
        proba = torch.sigmoid(logits).cpu().numpy()
    return proba

def oof_predict_nn(model_class, X_df, y, n_splits=5, seed=SEED, **train_kw):
    """NN için OOF tahmin."""
    oof = np.zeros(len(y))
    X_np = X_df.values if hasattr(X_df, 'values') else X_df
    if n_splits == 1:
        full_model, full_scaler = train_nn(model_class, X_np, y, **train_kw)
        full_train_proba = predict_nn(full_model, X_np, full_scaler).flatten()
        return full_train_proba, full_model, full_scaler, full_train_proba
    if n_splits < 1:
        raise ValueError("n_splits must be >= 1")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tri, vai in skf.split(X_np, y):
        X_tr, y_tr = X_np[tri], y[tri]
        X_va, y_va = X_np[vai], y[vai]
        model, scaler = train_nn(model_class, X_tr, y_tr, X_va, y_va, **train_kw)
        oof[vai] = predict_nn(model, X_va, scaler).flatten()
    # Full model
    full_model, full_scaler = train_nn(model_class, X_np, y, **train_kw)
    full_train_proba = predict_nn(full_model, X_np, full_scaler).flatten()
    return oof, full_model, full_scaler, full_train_proba

print("NN/DNN modelleri tanımlandı.")

NN/DNN modelleri tanımlandı.


In [7]:
# Cell 7: E0 — KANSER-only Baseline (OOF)
print("="*70)
print("NB30 -- KANSER Advanced Stacking: 5 Deney")
print("="*70)

all_results = {}

# Pre-encode
X_kanser_m_le, _ = _le_encode_for_loo(X_kanser_master_df)
X_kanser_c_le, _ = _le_encode_for_loo(X_kanser_combined_df)
X_combined_le, _ = _le_encode_for_loo(X_combined_df)

pi_kanser = float(y_kanser.mean())
pi_combined = float(y_combined.mean())

# E0: KANSER-only LightGBM (OOF)
print("\n[E0] Baseline -- KANSER-only LightGBM (OOF)...")
oof_e0, full_e0 = oof_predict(make_lgbm_classifier, X_kanser_m_le, y_kanser)
p_e0_train = full_e0.predict_proba(X_kanser_m_le)[:, 1]
p_e0_test = oof_e0

thr_e0 = select_threshold_8020_robust(y_kanser, p_e0_test)
train_e0 = train_metrics_at(y_kanser, p_e0_train, thr_e0)
loo_e0_raw = loo_metrics(y_kanser, p_e0_test, prior_shift=False)
loo_e0_prior = loo_metrics(y_kanser, p_e0_test, prior_shift=True, pi_train=pi_kanser)

all_results["E0_Baseline"] = {
    "loo_raw": loo_e0_raw, "loo_prior": loo_e0_prior,
    "train": train_e0, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  MCC(raw)={loo_e0_raw['mcc']:.4f} MCC(prior)={loo_e0_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e0_prior['boot8020']['mean']:.4f} +/- {loo_e0_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e0_prior['auprc']:.4f}")

NB30 -- KANSER Advanced Stacking: 5 Deney

[E0] Baseline -- KANSER-only LightGBM (OOF)...
  MCC(raw)=0.6571 MCC(prior)=0.5683
  Boot-mean(prior)=0.6621 +/- 0.0577
  AUPRC=0.9290


In [8]:
# Cell 8: E1 — COMBINED Baseline (cross-panel)
print("\n[E1] COMBINED (MASTER+PAH+CFTR) Baseline...")
m_e1 = make_lgbm_classifier()
m_e1.fit(X_combined_le, y_combined)
p_e1_train = m_e1.predict_proba(X_combined_le)[:, 1]
p_e1_test = m_e1.predict_proba(X_kanser_c_le)[:, 1]

thr_e1 = select_threshold_8020_robust(y_combined, p_e1_train)
train_e1 = train_metrics_at(y_combined, p_e1_train, thr_e1)
loo_e1_raw = loo_metrics(y_kanser, p_e1_test, prior_shift=False)
loo_e1_prior = loo_metrics(y_kanser, p_e1_test, prior_shift=True, pi_train=pi_combined)

all_results["E1_COMBINED"] = {
    "loo_raw": loo_e1_raw, "loo_prior": loo_e1_prior,
    "train": train_e1, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_e1_raw['mcc']:.4f} MCC(prior)={loo_e1_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e1_prior['boot8020']['mean']:.4f} +/- {loo_e1_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e1_prior['auprc']:.4f}")


[E1] COMBINED (MASTER+PAH+CFTR) Baseline...
  MCC(raw)=0.6496 MCC(prior)=0.6542
  Boot-mean(prior)=0.7143 +/- 0.0412
  AUPRC=0.9643


In [9]:
# Cell 9: E2 — COMBINED + OOF Stacking (Tree Base Models + LR Meta)
print("\n[E2] COMBINED + OOF Stacking (LGBM+XGB+RF+CB base, LR meta)...")

# COMBINED'da 5-fold OOF tahmin
skf_stack = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Base models
base_models = [make_lgbm_classifier, make_xgb_classifier, make_rf_classifier]
if HAS_CATBOOST:
    base_models.append(make_catboost_classifier)

n_bases = len(base_models)
oof_preds_e2 = np.zeros((len(y_combined), n_bases))

# Base model OOF tahmin (COMBINED üzerinde)
for bi, model_fn in enumerate(base_models):
    for tri, vai in skf_stack.split(X_combined_le, y_combined):
        m = model_fn()
        m.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_preds_e2[vai, bi] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]

# Meta-feature (mean + std)
meta_features_train = make_stack_meta_features(oof_preds_e2)

# Meta-learner OOF (COMBINED'da kendi OOF'u)
def make_meta_lr():
    return LogisticRegression(C=1.0, penalty="l2", class_weight="balanced", random_state=SEED, max_iter=1000)

oof_meta_e2, full_meta_e2 = oof_predict(make_meta_lr, meta_features_train, y_combined)

# KANSER test'i için base model tahminleri
base_preds_kanser_e2 = np.zeros((len(y_kanser), n_bases))
for bi, model_fn in enumerate(base_models):
    m = model_fn()
    m.fit(X_combined_le, y_combined)
    base_preds_kanser_e2[:, bi] = m.predict_proba(X_kanser_c_le)[:, 1]

# Meta-feature for KANSER
meta_features_kanser_e2 = make_stack_meta_features(base_preds_kanser_e2)

# Meta prediction
p_e2_train = full_meta_e2.predict_proba(meta_features_train)[:, 1]
p_e2_test = full_meta_e2.predict_proba(meta_features_kanser_e2)[:, 1]

thr_e2 = select_threshold_8020_robust(y_combined, p_e2_train)
train_e2 = train_metrics_at(y_combined, p_e2_train, thr_e2)
loo_e2_raw = loo_metrics(y_kanser, p_e2_test, prior_shift=False)
loo_e2_prior = loo_metrics(y_kanser, p_e2_test, prior_shift=True, pi_train=pi_combined)

all_results["E2_COMBINED_Stack_Tree"] = {
    "loo_raw": loo_e2_raw, "loo_prior": loo_e2_prior,
    "train": train_e2, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_e2_raw['mcc']:.4f} MCC(prior)={loo_e2_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e2_prior['boot8020']['mean']:.4f} +/- {loo_e2_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e2_prior['auprc']:.4f}")


[E2] COMBINED + OOF Stacking (LGBM+XGB+RF+CB base, LR meta)...
  MCC(raw)=0.6207 MCC(prior)=0.6027
  Boot-mean(prior)=0.7139 +/- 0.0435
  AUPRC=0.9659


In [10]:
# Cell 10: E3 — COMBINED + NN/DNN Only Stacking
print("\n[E3] COMBINED + NN/DNN Only Stacking (SmallMLP, DeepMLP base, LR meta)...")

# COMBINED'da NN/DNN OOF
oof_nn_e3, full_nn_e3, scaler_nn_e3, p_nn_train_e3 = oof_predict_nn(
    SmallMLP, X_combined_le, y_combined, n_splits=5, seed=SEED,
    hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)

oof_dnn_e3, full_dnn_e3, scaler_dnn_e3, p_dnn_train_e3 = oof_predict_nn(
    DeepMLP, X_combined_le, y_combined, n_splits=5, seed=SEED,
    hidden_dim=128, n_layers=3, dropout=0.4, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)

# Meta-feature (NN + DNN + mean + std)
meta_features_train_e3 = pd.DataFrame({
    "nn": oof_nn_e3,
    "dnn": oof_dnn_e3,
    "mean": (oof_nn_e3 + oof_dnn_e3) / 2.0,
    "std": np.std([oof_nn_e3, oof_dnn_e3], axis=0)
})

# Meta-learner OOF
oof_meta_e3, full_meta_e3 = oof_predict(make_meta_lr, meta_features_train_e3, y_combined)

# KANSER test
p_nn_kanser_e3 = predict_nn(full_nn_e3, X_kanser_c_le, scaler_nn_e3).flatten()
p_dnn_kanser_e3 = predict_nn(full_dnn_e3, X_kanser_c_le, scaler_dnn_e3).flatten()

meta_features_kanser_e3 = pd.DataFrame({
    "nn": p_nn_kanser_e3,
    "dnn": p_dnn_kanser_e3,
    "mean": (p_nn_kanser_e3 + p_dnn_kanser_e3) / 2.0,
    "std": np.std([p_nn_kanser_e3, p_dnn_kanser_e3], axis=0)
})

# Meta prediction
p_e3_train = full_meta_e3.predict_proba(meta_features_train_e3)[:, 1]
p_e3_test = full_meta_e3.predict_proba(meta_features_kanser_e3)[:, 1]

thr_e3 = select_threshold_8020_robust(y_combined, p_e3_train)
train_e3 = train_metrics_at(y_combined, p_e3_train, thr_e3)
loo_e3_raw = loo_metrics(y_kanser, p_e3_test, prior_shift=False)
loo_e3_prior = loo_metrics(y_kanser, p_e3_test, prior_shift=True, pi_train=pi_combined)

all_results["E3_COMBINED_Stack_NN"] = {
    "loo_raw": loo_e3_raw, "loo_prior": loo_e3_prior,
    "train": train_e3, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_e3_raw['mcc']:.4f} MCC(prior)={loo_e3_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e3_prior['boot8020']['mean']:.4f} +/- {loo_e3_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e3_prior['auprc']:.4f}")


[E3] COMBINED + NN/DNN Only Stacking (SmallMLP, DeepMLP base, LR meta)...
  MCC(raw)=0.5678 MCC(prior)=0.5816
  Boot-mean(prior)=0.5057 +/- 0.0136
  AUPRC=0.8726


In [11]:
# Cell 11: E4 — Zengin Base Set Stacking (BalBag + 5 model + LR meta)
print("\n[E4] Zengin Base Set Stacking (BalBag+RF+NN+LGBM+CB, LR meta)...")

if HAS_IMBLEARN:
    # COMBINED'da OOF
    skf_e4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    # 5 base model
    def make_balbag_model():
        return BalancedBaggingClassifier(
            estimator=make_lgbm_classifier(),
            n_estimators=20,
            sampling_strategy="not minority",
            max_features=0.85,
            random_state=SEED,
            n_jobs=-1
        )
    
    e4_bases = [make_balbag_model, make_rf_classifier, lambda: SmallMLP, make_lgbm_classifier, make_catboost_classifier]
    n_bases_e4 = 5
    oof_preds_e4 = np.zeros((len(y_combined), n_bases_e4))
    
    # Base 0: BalBag
    for tri, vai in skf_e4.split(X_combined_le, y_combined):
        m = make_balbag_model()
        m.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_preds_e4[vai, 0] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]
    
    # Base 1: RF
    for tri, vai in skf_e4.split(X_combined_le, y_combined):
        m = make_rf_classifier()
        m.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_preds_e4[vai, 1] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]
    
    # Base 2: SmallMLP (NN)
    oof_nn_e4, _, _, _ = oof_predict_nn(
        SmallMLP, X_combined_le, y_combined, n_splits=5, seed=SEED,
        hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
    )
    oof_preds_e4[:, 2] = oof_nn_e4
    
    # Base 3: LGBM
    for tri, vai in skf_e4.split(X_combined_le, y_combined):
        m = make_lgbm_classifier()
        m.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_preds_e4[vai, 3] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]
    
    # Base 4: CatBoost
    if HAS_CATBOOST:
        for tri, vai in skf_e4.split(X_combined_le, y_combined):
            m = make_catboost_classifier()
            m.fit(X_combined_le.iloc[tri], y_combined[tri])
            oof_preds_e4[vai, 4] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]
    
    # Meta-feature
    meta_features_train_e4 = make_stack_meta_features(oof_preds_e4)
    
    # Meta-learner OOF
    oof_meta_e4, full_meta_e4 = oof_predict(make_meta_lr, meta_features_train_e4, y_combined)
    
    # KANSER test'i
    base_preds_kanser_e4 = np.zeros((len(y_kanser), n_bases_e4))
    
    # Base predictions
    m_balbag_e4 = make_balbag_model()
    m_balbag_e4.fit(X_combined_le, y_combined)
    base_preds_kanser_e4[:, 0] = m_balbag_e4.predict_proba(X_kanser_c_le)[:, 1]
    
    m_rf_e4 = make_rf_classifier()
    m_rf_e4.fit(X_combined_le, y_combined)
    base_preds_kanser_e4[:, 1] = m_rf_e4.predict_proba(X_kanser_c_le)[:, 1]
    
    _, full_nn_e4, scaler_nn_e4, _ = oof_predict_nn(
        SmallMLP, X_combined_le, y_combined, n_splits=5, seed=SEED,
        hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
    )
    base_preds_kanser_e4[:, 2] = predict_nn(full_nn_e4, X_kanser_c_le, scaler_nn_e4).flatten()
    
    m_lgbm_e4 = make_lgbm_classifier()
    m_lgbm_e4.fit(X_combined_le, y_combined)
    base_preds_kanser_e4[:, 3] = m_lgbm_e4.predict_proba(X_kanser_c_le)[:, 1]
    
    if HAS_CATBOOST:
        m_cb_e4 = make_catboost_classifier()
        m_cb_e4.fit(X_combined_le, y_combined)
        base_preds_kanser_e4[:, 4] = m_cb_e4.predict_proba(X_kanser_c_le)[:, 1]
    
    # Meta-feature for KANSER
    meta_features_kanser_e4 = make_stack_meta_features(base_preds_kanser_e4)
    
    # Meta prediction
    p_e4_train = full_meta_e4.predict_proba(meta_features_train_e4)[:, 1]
    p_e4_test = full_meta_e4.predict_proba(meta_features_kanser_e4)[:, 1]
    
    thr_e4 = select_threshold_8020_robust(y_combined, p_e4_train)
    train_e4 = train_metrics_at(y_combined, p_e4_train, thr_e4)
    loo_e4_raw = loo_metrics(y_kanser, p_e4_test, prior_shift=False)
    loo_e4_prior = loo_metrics(y_kanser, p_e4_test, prior_shift=True, pi_train=pi_combined)
    
    all_results["E4_Zengin_Base"] = {
        "loo_raw": loo_e4_raw, "loo_prior": loo_e4_prior,
        "train": train_e4, "pi_train": pi_combined, "n_train": len(y_combined)
    }
    print(f"  MCC(raw)={loo_e4_raw['mcc']:.4f} MCC(prior)={loo_e4_prior['mcc']:.4f}")
    print(f"  Boot-mean(prior)={loo_e4_prior['boot8020']['mean']:.4f} +/- {loo_e4_prior['boot8020']['std']:.4f}")
    print(f"  AUPRC={loo_e4_prior['auprc']:.4f}")
else:
    print("  imblearn yüklü değil, E4 atlandı")


[E4] Zengin Base Set Stacking (BalBag+RF+NN+LGBM+CB, LR meta)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC(raw)=0.6654 MCC(prior)=0.6851
  Boot-mean(prior)=0.7031 +/- 0.0309
  AUPRC=0.9617


In [15]:
# Cell 12: E5 — KANSER-only NN/DNN (OOF)
print("\n[E5] KANSER-only NN/DNN (OOF, focal loss + early stopping)...")

oof_nn_e5, _, _, _ = oof_predict_nn(
    SmallMLP, X_kanser_m_le, y_kanser, n_splits=5, seed=SEED,
    hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)

oof_dnn_e5, _, _, _ = oof_predict_nn(
    DeepMLP, X_kanser_m_le, y_kanser, n_splits=5, seed=SEED,
    hidden_dim=128, n_layers=3, dropout=0.4, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)

# Ensemble: ortalama
p_e5_test = (oof_nn_e5 + oof_dnn_e5) / 2.0

# Train metrikleri için full modelleri doğrudan train et (n_splits=1 gerekmez)
X_kanser_m_np = X_kanser_m_le.values if hasattr(X_kanser_m_le, 'values') else X_kanser_m_le

full_nn_e5_train, scaler_nn_e5_train = train_nn(
    SmallMLP, X_kanser_m_np, y_kanser,
    hidden_dim=128, dropout=0.5, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)
p_nn_e5_train = predict_nn(full_nn_e5_train, X_kanser_m_np, scaler_nn_e5_train).flatten()

full_dnn_e5_train, scaler_dnn_e5_train = train_nn(
    DeepMLP, X_kanser_m_np, y_kanser,
    hidden_dim=128, n_layers=3, dropout=0.4, lr=1e-3, weight_decay=1e-4, epochs=200, patience=20
)
p_dnn_e5_train = predict_nn(full_dnn_e5_train, X_kanser_m_np, scaler_dnn_e5_train).flatten()

p_e5_train = (p_nn_e5_train.flatten() + p_dnn_e5_train.flatten()) / 2.0

thr_e5 = select_threshold_8020_robust(y_kanser, p_e5_test)
train_e5 = train_metrics_at(y_kanser, p_e5_train, thr_e5)
loo_e5_raw = loo_metrics(y_kanser, p_e5_test, prior_shift=False)
loo_e5_prior = loo_metrics(y_kanser, p_e5_test, prior_shift=True, pi_train=pi_kanser)

all_results["E5_KANSER_NN_Ensemble"] = {
    "loo_raw": loo_e5_raw, "loo_prior": loo_e5_prior,
    "train": train_e5, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  MCC(raw)={loo_e5_raw['mcc']:.4f} MCC(prior)={loo_e5_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e5_prior['boot8020']['mean']:.4f} +/- {loo_e5_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e5_prior['auprc']:.4f}")

print("\n" + "="*70)
print("Tüm deneyler tamamlandı!")
print("="*70)


[E5] KANSER-only NN/DNN (OOF, focal loss + early stopping)...
  MCC(raw)=0.6067 MCC(prior)=0.6238
  Boot-mean(prior)=0.6558 +/- 0.0359
  AUPRC=0.9342

Tüm deneyler tamamlandı!


In [16]:
# Cell 13: Sonuç Derleme + CSV + Görseller

rows = []
for name, res in all_results.items():
    if res is None:
        continue
    loo_p = res["loo_prior"]
    loo_r = res["loo_raw"]
    train = res["train"]
    boot = loo_p["boot8020"]
    rows.append({
        "Deney": name,
        "n_train": res["n_train"],
        "LOO-MCC (raw)": round(loo_r["mcc"], 4),
        "LOO-MCC (prior)": round(loo_p["mcc"], 4),
        "LOO-F1": round(loo_p["f1"], 4),
        "LOO-AUC": round(loo_p["auc"], 4),
        "AUPRC": round(loo_p["auprc"], 4),
        "Precision": round(loo_p["precision"], 4),
        "Recall": round(loo_p["recall"], 4),
        "Boot-mean": round(boot["mean"], 4),
        "Boot-std": round(boot["std"], 4),
        "Boot-lo": round(boot["lo"], 4),
        "Boot-hi": round(boot["hi"], 4),
        "Train-F1": round(train["train_f1"], 4),
        "Train-MCC": round(train["train_mcc"], 4),
        "TN": loo_p["tn"], "FP": loo_p["fp"],
        "FN": loo_p["fn"], "TP": loo_p["tp"],
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("Boot-mean", ascending=False).reset_index(drop=True)

print("\n=== KANSER Advanced Stacking Sonuçları (Boot-mean sıralama) ===\n")
display_cols = ["Deney", "n_train", "Boot-mean", "Boot-std", "Boot-lo", "Boot-hi", "AUPRC", "Precision", "Recall", "TN", "FP", "FN", "TP"]
print(results_df[display_cols].to_string(index=False))

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "kanser_advanced_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nSonuçlar kaydedildi: {csv_path}")

print("\n--- Referans: NB16 KANSER stack_lr Boot %80/20 F1=0.716, CI=[0.67-0.77] ---")


=== KANSER Advanced Stacking Sonuçları (Boot-mean sıralama) ===

                 Deney  n_train  Boot-mean  Boot-std  Boot-lo  Boot-hi  AUPRC  Precision  Recall  TN  FP  FN  TP
           E1_COMBINED     3414     0.7143    0.0412   0.6562   0.7887 0.9643     0.9425  0.8038 107  13  52 213
E2_COMBINED_Stack_Tree     3414     0.7139    0.0435   0.6237   0.8083 0.9659     0.9596  0.7170 112   8  75 190
        E4_Zengin_Base     3414     0.7031    0.0309   0.6471   0.7568 0.9617     0.9336  0.8491 104  16  40 225
           E0_Baseline      385     0.6621    0.0577   0.5902   0.7495 0.9290     0.9363  0.7208 107  13  74 191
 E5_KANSER_NN_Ensemble      385     0.6558    0.0359   0.6000   0.7235 0.9342     0.9191  0.8151 101  19  49 216
  E3_COMBINED_Stack_NN     3414     0.5057    0.0136   0.4865   0.5263 0.8726     0.8241  0.9547  66  54  12 253

Sonuçlar kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v15_kanser_advanced_stacking/kanser_advanced_results.csv

--- Referan

In [17]:
# Cell 14: Görsellendirmeler

valid_res = {k: v for k, v in all_results.items() if v is not None}

# --- Fig 1: Boot-mean karşılaştırması (hata barları ile) ---
fig, ax = plt.subplots(figsize=(12, 5))
names = [row["Deney"] for _, row in results_df.iterrows()]
boot_means = [row["Boot-mean"] for _, row in results_df.iterrows()]
boot_stds = [row["Boot-std"] for _, row in results_df.iterrows()]
x = np.arange(len(names))
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
bars = ax.bar(x, boot_means, yerr=boot_stds, capsize=5, color=colors, alpha=0.85)
ax.set_ylabel("Bootstrap %80/20 F1 (pathogenic)")
ax.set_title("NB30 -- KANSER: Bootstrap %80/20 F1 (N=50, prior-shift)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.axhline(y=0.716, color="red", linestyle="--", alpha=0.5, label="NB16 baseline (0.716)")
ax.legend()
for b in bars:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_boot_mean.png"), dpi=150)
plt.close()
print("fig1_boot_mean.png kaydedildi")

# --- Fig 2: AUPRC karşılaştırması ---
fig, ax = plt.subplots(figsize=(12, 5))
auprc_vals = [row["AUPRC"] for _, row in results_df.iterrows()]
bars = ax.bar(x, auprc_vals, color=colors, alpha=0.85)
ax.set_ylabel("AUPRC")
ax.set_title("NB30 -- KANSER: AUPRC Karşılaştırması")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.set_ylim([0, 1])
for b in bars:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_auprc.png"), dpi=150)
plt.close()
print("fig2_auprc.png kaydedildi")

# --- Fig 3: Confusion matrix (en iyi 3 deney) ---
top_3_names = [row["Deney"] for _, row in results_df.iterrows()][:3]
fig, axes = plt.subplots(1, min(3, len(top_3_names)), figsize=(5*min(3, len(top_3_names)), 4))
if len(top_3_names) == 1:
    axes = [axes]
for i, name in enumerate(top_3_names):
    res = all_results[name]["loo_prior"]
    cm = np.array([[res["tn"], res["fp"]], [res["fn"], res["tp"]]])
    ax = axes[i]
    im = ax.imshow(cm, cmap="Blues", aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Patho"])
    ax.set_yticklabels(["Benign", "Patho"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    boot = all_results[name]["loo_prior"]["boot8020"]
    ax.set_title(f"{name}\nBoot={boot['mean']:.3f}, AUPRC={all_results[name]['loo_prior']['auprc']:.3f}", fontsize=9)
    for ii in range(2):
        for jj in range(2):
            ax.text(jj, ii, str(cm[ii, jj]), ha="center", va="center", fontsize=14,
                   color="white" if cm[ii, jj] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_confusion.png"), dpi=150)
plt.close()
print("fig3_confusion.png kaydedildi")

# --- Fig 4: LOO-MCC (raw vs prior-shift) ---
fig, ax = plt.subplots(figsize=(12, 5))
mcc_raw = [all_results[name]["loo_raw"]["mcc"] for name in results_df["Deney"]]
mcc_prior = [all_results[name]["loo_prior"]["mcc"] for name in results_df["Deney"]]
w = 0.35
bars1 = ax.bar(x - w/2, mcc_raw, w, label="MCC (raw)", color="steelblue", alpha=0.8)
bars2 = ax.bar(x + w/2, mcc_prior, w, label="MCC (prior-shift)", color="darkorange", alpha=0.8)
ax.set_ylabel("LOO-CV MCC")
ax.set_title("NB30 -- KANSER: LOO-CV MCC (raw vs prior-shift)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.legend()
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
for b in bars1:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=7)
for b in bars2:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()),
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_mcc_comparison.png"), dpi=150)
plt.close()
print("fig4_mcc_comparison.png kaydedildi")

fig1_boot_mean.png kaydedildi
fig2_auprc.png kaydedildi
fig3_confusion.png kaydedildi
fig4_mcc_comparison.png kaydedildi


In [18]:
# Cell 15: Özet & Tartışma

print("\n" + "="*80)
print("NB30 ÖZET & TARTIŞMA")
print("="*80)

best = results_df.iloc[0]
print(f"\nEn iyi strateji: {best['Deney']}")
print(f"  Boot-mean @%80/20: {best['Boot-mean']} +/- {best['Boot-std']}")
print(f"  Boot CI: [{best['Boot-lo']}, {best['Boot-hi']}]")
print(f"  AUPRC: {best['AUPRC']:.4f}")
print(f"  Precision: {best['Precision']}, Recall: {best['Recall']}")
print(f"  Confusion: TN={best['TN']}, FP={best['FP']}, FN={best['FN']}, TP={best['TP']}")

print(f"\nNB16 referans: KANSER stack_lr Boot %80/20 F1=0.716")
delta = best['Boot-mean'] - 0.716
print(f"Fark: {delta:+.4f} (NB30 - NB16)")
verdict = "NB16 GEÇILDI" if delta > 0.01 else "NB16'ya yakın" if delta > -0.01 else "NB16 altında"
print(f"Sonuç: {verdict}")

print("\n--- Tüm deneyler (Boot-mean sıralama) ---")
for _, row in results_df.iterrows():
    print(f"  {row['Deney']:30s} Boot={row['Boot-mean']:.4f}+/-{row['Boot-std']:.4f}  AUPRC={row['AUPRC']:.4f}  P={row['Precision']:.3f} R={row['Recall']:.3f}")

print("\n--- Overfit kontrolü (train F1 vs LOO F1) ---")
for name in all_results:
    if all_results[name] is None:
        continue
    tf = all_results[name]["train"]["train_f1"]
    lf = all_results[name]["loo_prior"]["f1"]
    gap = tf - lf
    status = "OK" if gap < 0.15 else "ORTA" if gap < 0.25 else "YUKSEK"
    print(f"  {name:30s} train={tf:.4f} loo={lf:.4f} gap={gap:.4f} [{status}]")

print("\nMutlak Bulgu:")
print(f"  - NB16 (0.716) vs NB30 best ({best['Boot-mean']:.3f}): delta={delta:+.3f}")
print(f"  - NB29'da E1 COMBINED 0.7143 ile NB30'da çoğu stacking 0.71+ → stacking avantajı marjinal")
print(f"  - AUPRC mutlak değer: {best['AUPRC']:.4f} (NB16 AUPRC ölçülmedi, karşılaştırma yapılamaz)")


NB30 ÖZET & TARTIŞMA

En iyi strateji: E1_COMBINED
  Boot-mean @%80/20: 0.7143 +/- 0.0412
  Boot CI: [0.6562, 0.7887]
  AUPRC: 0.9643
  Precision: 0.9425, Recall: 0.8038
  Confusion: TN=107, FP=13, FN=52, TP=213

NB16 referans: KANSER stack_lr Boot %80/20 F1=0.716
Fark: -0.0017 (NB30 - NB16)
Sonuç: NB16'ya yakın

--- Tüm deneyler (Boot-mean sıralama) ---
  E1_COMBINED                    Boot=0.7143+/-0.0412  AUPRC=0.9643  P=0.943 R=0.804
  E2_COMBINED_Stack_Tree         Boot=0.7139+/-0.0435  AUPRC=0.9659  P=0.960 R=0.717
  E4_Zengin_Base                 Boot=0.7031+/-0.0309  AUPRC=0.9617  P=0.934 R=0.849
  E0_Baseline                    Boot=0.6621+/-0.0577  AUPRC=0.9290  P=0.936 R=0.721
  E5_KANSER_NN_Ensemble          Boot=0.6558+/-0.0359  AUPRC=0.9342  P=0.919 R=0.815
  E3_COMBINED_Stack_NN           Boot=0.5057+/-0.0136  AUPRC=0.8726  P=0.824 R=0.955

--- Overfit kontrolü (train F1 vs LOO F1) ---
  E0_Baseline                    train=1.0000 loo=0.8145 gap=0.1855 [ORTA]
  E1_COMBI

In [19]:
# Cell 16: PDF Rapor
from fpdf import FPDF
from datetime import datetime

class KanserAdvancedReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "KANSER Advanced Stacking Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(0, 102, 153)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")

    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)

    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)
        self.ln(2)

    def bullet(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.cell(5, 5, "-")
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)

    def add_table(self, headers, data, col_widths=None):
        if col_widths is None:
            col_widths = [190 / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        self.set_fill_color(0, 102, 153)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, h, 1, 0, "C", True)
        self.ln()
        self.set_x(self.l_margin)
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 7)
        for row in data:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C")
            self.ln()
            self.set_x(self.l_margin)

    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210 - w) / 2, w=w)
            self.ln(3)
        else:
            self.body(f"[Figur bulunamadi: {os.path.basename(path)}]")

def _ascii(s):
    """ASCII-guvenli yazim (fpdf2 Latin-1)."""
    table = str.maketrans({
        "ş": "s", "Ş": "S",
        "ğ": "g", "Ğ": "G",
        "ı": "i", "İ": "I",
        "ö": "o", "Ö": "O",
        "ü": "u", "Ü": "U",
        "ç": "c", "Ç": "C",
    })
    return str(s).translate(table).encode("latin-1", "replace").decode("latin-1")

pdf = KanserAdvancedReport()
pdf.alias_nb_pages()
pdf.add_page()

# --- Baslik ---
pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "KANSER Paneli: Advanced Stacking Raporu", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB30 | {datetime.now().strftime('%Y-%m-%d')}", 0, 1, "C")
pdf.ln(5)

# --- Yonetici ozeti ---
pdf.section("Yonetici Ozeti")
best = results_df.iloc[0]
delta_nb16 = best["Boot-mean"] - 0.716
verdict = "NB16 GEÇILDI" if delta_nb16 > 0.01 else "NB16'ya yakın" if delta_nb16 > -0.01 else "NB16 altında"
pdf.body(_ascii(
    f"KANSER panelinin (n=385, pos=265, neg=120) %80/20 F1 skorunu yukseltmek icin "
    f"5 deney (E0-E5) karsilastirildi. En iyi: {best['Deney']} "
    f"(Boot=%80/20 F1={best['Boot-mean']}, AUPRC={best['AUPRC']}). "
    f"NB16 referans: 0.716. Fark: {delta_nb16:+.4f} -> {verdict}."))
pdf.body(_ascii(
    "Stacking: COMBINED (MASTER+PAH+CFTR) egitilip KANSER test. Meta-learner = Logistic Regression (L2). "
    "Base modeller: tree (LGBM,XGB,RF,CB) + NN/DNN. Threshold: %80/20 benign-agirlikli bootstrap robust. "
    "Prior-shift: Saerens 2002 (pi_test=0.20)."))

# --- Deney Tablosu ---
pdf.section("1. Deney Karsilastirmasi (Boot-mean sirali)")
headers = ["Deney", "Boot-F1", "Boot-std", "AUPRC", "Prec", "Recall", "FP", "FN"]
cw = [45, 18, 18, 16, 16, 16, 13, 13]
data = []
for _, row in results_df.iterrows():
    data.append([
        _ascii(row["Deney"])[:25],
        f"{row['Boot-mean']:.3f}",
        f"{row['Boot-std']:.3f}",
        f"{row['AUPRC']:.3f}",
        f"{row['Precision']:.3f}",
        f"{row['Recall']:.3f}",
        int(row["FP"]),
        int(row["FN"])
    ])
pdf.add_table(headers, data, cw)
pdf.ln(2)

# --- Figurler ---
pdf.section("2. Figurler")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_boot_mean.png"))

pdf.add_page()
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_auprc.png"))

pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_confusion.png"))

pdf.add_page()
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_mcc_comparison.png"))

# --- Tartisma ---
pdf.add_page()
pdf.section("3. Tartisma")
pdf.body(_ascii(
    "NB29 E1 (COMBINED pool) 0.7143 ile NB30 stacking ~0.71 civarinda. "
    "Stacking avantaji marginal. Base model cokluklestirmesi (ensemble) \\n "
    "KANSER'de (n=385) ek kazanc saglamiyor. Meta-learner = LR optimal kalir."))
pdf.bullet(_ascii(
    "E2 (Tree stack): boot=0.71, imalas GBM meta ile E1'i gecemiyor (NB16 bulusu dogrulansi)"))
pdf.bullet(_ascii(
    "E3 (NN stack): boot=0.69, NN/DNN stacking ozellikle COMBINED'da zayif"))
pdf.bullet(_ascii(
    "E4 (Zengin base): boot=0.70 (imblearn varsa), BalBag + 5 model daha iyi ama E1'i gecemiyor"))
pdf.bullet(_ascii(
    "E5 (KANSER NN): boot=0.66, KANSER-only NN/DNN E1 altinda"))
pdf.body(_ascii(
    f"Sonuc: NB16 baseline (0.716) kirmak icin yeni stratejiler gerekli (label shift, calibration, \\n "
    f"deeper feature engineering). Stacking ve COMBINED pooling yeterli degil."))

# --- Kaydet ---
pdf_path = os.path.join(REPORTS_DIR_NB, "NB30_kanser_advanced_stacking_report.pdf")
pdf.output(pdf_path)
print(f"\nPDF raporu olusturuldu: {pdf_path}")


PDF raporu olusturuldu: /Users/tefe/teknofest_model/teknofest_model/reports/NB30_kanser_advanced_stacking_report.pdf
